# Identify Next Batch

 ![Incremental Data Processing](../../z-course-images/formula1-incremental-data-processing.png "Incremental Data Processing")

**This code is part of an incremental batch-processing pipeline in Databricks. It checks which batch folders are available in the Landing layer, compares them with the batches recorded in a control table, and selects the earliest unprocessed batch to process next.**

- For example, if Landing contains batches batch_001, batch_002, and batch_003, and the control table shows that batch_001 and batch_002 are already being processed or completed, the code selects batch_003.

![image_1792337917392.png](./image_1792337917392.png "image_1792337917392.png")

In [0]:
%run ../00-common/01.environment_config

In [0]:
control_table = f"{catalog_name}.{control_schema}.batch_control"

In [0]:
control_table

In [0]:
from pyspark.sql.functions import *

In [0]:
file_path=dbutils.fs.ls(landing_folder_path)
landing_batches=[]
for files in file_path:
    if files.isDir():
        files_n = files.name.rstrip("/")
        landing_batches.append(files_n)

landing_batches = sorted(landing_batches)

In [0]:
landing_batches

In [0]:
if spark.catalog.tableExists(control_table):
    tracked_batches=[row.batch_id
    for row in (spark.table(control_table)
                .filter(col("status").isin('completed','in_progress'))
                .select('batch_id')
                .distinct()
                .collect()
                )
    ]
else:
    tracked_batches = []

In [0]:
#sorted(list(set(landing_batches) - set(tracked_batches)))

In [0]:
new_batches = sorted(list(set(landing_batches) - set(tracked_batches)))
next_batch = new_batches[0] if new_batches else None
print(f"Landing batches     : {landing_batches}")
print(f"Tracked batches     : {tracked_batches}")
print(f"Next batch to process: {next_batch}")



In [0]:
if next_batch is None:
    dbutils.jobs.taskValues.set(key="p_batch_id", value="")
    dbutils.jobs.taskValues.set(key="has_batch", value="false")
else:
    dbutils.jobs.taskValues.set(key="p_batch_id", value=next_batch)
    dbutils.jobs.taskValues.set(key="has_batch", value="true")